# Checking how a record was built, before decoding it

A decoder result is a statement about a record. If the record was assembled wrong, the result is
wrong by a factor and nothing downstream can tell: the detectors build, the decoder runs, the
logical error rate comes out, and the plot looks like every other plot.

`qb_compiler.record` holds eight checks for the ways that happens. This notebook builds a
repetition code memory record in software, runs the checks on it, then breaks the record in the
way a real one was broken and watches the checks find it.

Everything here is made inside the notebook. No data is needed and nothing downloads. The last
section runs against a real record if one is on the machine, and says so politely if not.

## 1. A record to look at

In [1]:
import numpy as np

from qb_compiler.record import FIRST_LAYER_PREMISE, RecordSpec, validate
from qb_compiler.record.dem import build_repetition_dem, decode_records


def simulate(d=5, rounds=5, shots=8000, p_data=0.04, p_meas=0.04, seed=0):
    """Walk a repetition code memory forward in time and keep what the hardware would store."""
    rng = np.random.default_rng(seed)
    state = np.zeros((shots, d), dtype=np.uint8)
    stored = np.zeros((shots, rounds, d - 1), dtype=np.uint8)
    for t in range(rounds):
        state ^= (rng.random((shots, d)) < p_data).astype(np.uint8)
        parity = state[:, :-1] ^ state[:, 1:]
        stored[:, t] = parity ^ (rng.random((shots, d - 1)) < p_meas).astype(np.uint8)
    state ^= (rng.random((shots, d)) < p_data).astype(np.uint8)
    final_parity = (state[:, :-1] ^ state[:, 1:]).astype(np.uint8)
    prepared = (rng.random(shots) < 0.5).astype(np.uint8)
    return {
        "stored": stored,
        "final_parity": final_parity,
        "labels": state[:, 0].astype(np.uint8),
        "prepared": prepared,
        "d": d,
        "rounds": rounds,
    }


def build(run, reverse_rounds):
    """Rebuild detectors from the stored blocks, the way a loader does."""
    stored = run["stored"][:, ::-1, :] if reverse_rounds else run["stored"]
    stored = np.ascontiguousarray(stored)
    shots, rounds, sites = stored.shape
    detectors = np.empty((shots, rounds + 1, sites), dtype=np.uint8)
    detectors[:, 0] = stored[:, 0]
    detectors[:, 1:rounds] = stored[:, 1:] ^ stored[:, :-1]
    detectors[:, rounds] = run["final_parity"] ^ stored[:, rounds - 1]
    return RecordSpec(
        detectors=detectors,
        labels=run["labels"],
        raw_syndromes=stored,
        final_parity=run["final_parity"],
        logical_state=run["prepared"],
        dem=build_repetition_dem(run["d"], run["rounds"], p_data=0.04, p_meas=0.04),
        # This run differences its first round against the prepared state, so that round
        # is quiet. The round profile check reads that premise off the record instead of
        # assuming it, because it is false where preparation is the noisy part.
        meta={FIRST_LAYER_PREMISE: {"source": "the generator above"}},
    )


run = simulate()
spec = build(run, reverse_rounds=False)
served = decode_records(spec.dem, spec.detector_matrix())
print(f"shots {spec.n_shots}, rounds {spec.n_rounds}, sites {spec.n_sites}")
print(f"detectors {spec.n_detectors}, mechanisms {spec.dem.n_mechanisms}")
print(f"logical error rate {(served != spec.labels).mean():.4f}")

shots 8000, rounds 6, sites 4
detectors 24, mechanisms 50
logical error rate 0.0170


## 2. Every check, listed

`validate` runs all eight and reports each one. A check that cannot run on this record says which
field it wanted rather than quietly passing. Three of them can be critical: the event density
always, the round profile on a record that declares a quiet first layer, and the model
fingerprint when an expected one is supplied. `passed` is
the critical checks passing. The rest are advisory: they are reported, and they do not vote,
because several of them have records on which they have no power and a check with nothing to read
should not decide a verdict.

In [2]:
def table(report):
    print(f"passed: {report.passed}")
    print(f"{'check':<22} {'status':<7} {'critical':<9} value")
    print("-" * 78)
    for item in report.checks:
        value = ""
        if item.name == "round_profile" and "first_round_ratio" in item.measured:
            value = (
                f"first {item.measured['first_round_rate']:.4f} "
                f"({item.measured['first_round_ratio']:.2f} times the median)"
            )
        elif item.name == "endpoint_agreement" and "ratio" in item.measured:
            value = f"last over first {item.measured['ratio']:.2f}"
        elif item.name == "event_density":
            value = f"rate {item.measured['event_rate']:.4f}"
        elif item.name == "time_mirror_control" and "logical_error_rate" in item.measured:
            value = (
                f"{item.measured['logical_error_rate']:.4f} against "
                f"{item.measured['mirrored_logical_error_rate']:.4f} reversed"
            )
        elif item.name == "state_profile" and "min_step" in item.measured:
            value = f"smallest step {item.measured['min_step']:+.4f}"
        elif item.name == "dem_fingerprint" and "n_mechanisms" in item.measured:
            value = f"{item.measured['n_mechanisms']} mechanisms"
        if not value:
            value = item.detail[:44]
        print(f"{item.name:<22} {item.status:<7} {str(item.critical):<9} {value}")


report = validate(spec)
table(report)

passed: True
check                  status  critical  value
------------------------------------------------------------------------------
round_profile          PASS    True      first 0.1082 (0.77 times the median)
endpoint_agreement     PASS    False     last over first 0.36
event_density          PASS    True      rate 0.1296
type_consistency       SKIP    False     needs deterministic_sites naming which stabi
time_mirror_control    PASS    False     0.0170 against 0.0179 reversed
state_profile          PASS    False     smallest step +0.0336
dem_fingerprint        PASS    False     50 mechanisms
label_reconstruction   SKIP    False     needs final_data; not in this record


The thresholds each check compared against are in the report, not just in the source.

The round profile reads one more thing off the record: whether this platform's first
layer is quiet, declared in `meta["first_layer_premise"]` and set by the loaders that
know their platform. The rule compares the first layer against the steady median, which
only separates a fault where that premise holds, so a record that declares none has its
layer profile reported and nothing decided on it.

In [3]:
for name in ("round_profile", "endpoint_agreement", "event_density", "state_profile"):
    print(f"{name:<22} {report.check(name).threshold}")

round_profile          {'first_round_max_ratio': 0.9, 'last_round_max_ratio': 1.5}
endpoint_agreement     {'max_ratio': 2.0, 'no_power_low': 0.8, 'no_power_high': 1.25}
event_density          {'low': 0.005, 'high': 0.35}
state_profile          {'min_step': -0.005}


## 3. The same run with its rounds stored backwards

This is a real fault, not an invented one. A hardware register can store its rounds last round
first, and reshaping it without reversing runs the experiment backwards in time. Nothing raises.
The same shots, read the wrong way round, look like this.

In [4]:
mirrored = build(run, reverse_rounds=True)
bad = validate(mirrored)
table(bad)

passed: False
check                  status  critical  value
------------------------------------------------------------------------------
round_profile          FAIL    True      first 0.2984 (2.13 times the median)
endpoint_agreement     FAIL    False     last over first 2.76
event_density          PASS    True      rate 0.1929
type_consistency       SKIP    False     needs deterministic_sites naming which stabi
time_mirror_control    PASS    False     0.0457 against 0.0446 reversed
state_profile          FAIL    False     smallest step -0.0623
dem_fingerprint        PASS    False     50 mechanisms
label_reconstruction   SKIP    False     needs final_data; not in this record


Three checks caught it, from three directions, and each says what it saw.

In [5]:
for name in ("round_profile", "endpoint_agreement", "state_profile"):
    print(name)
    print(" ", bad.check(name).detail)
    print()

round_profile
  the first round fires at 0.2984, 2.13x the steady median 0.1399, above the 0.9x limit this record declares. A first round differences against a reset and should fire at roughly half the steady rate; firing at or above it is what a record stored last round first looks like; the last round fires at 0.2974, 2.13x the steady median 0.1399, above the 1.5x limit this record declares. The last round differences the final data parity against the last stabilizer round, so it should be quieter than a stabilizer round, not louder

endpoint_agreement
  the final parity mismatches the round this loader treats as LAST 0.2974 of the time, against 0.1079 for the round it treats as FIRST (2.76x, limit 2.0x). The final data readout happens at the end of the run, so it should not disagree with the round nearest the end far more than with the far one. That is what rounds stored last round first look like: try loading with the round order reversed

state_profile
  on chains prepared in the 

## 4. What the time mirror control tests

The time mirror control reverses the detector axis and decodes again with the error model
unchanged, then compares the two decodes shot by shot. It is advisory: it never decides the
verdict.

That is a different fault from rounds stored backwards, and on a uniform repetition memory it has
little to find, because such a record is nearly symmetric in time: a fresh first round and a final
data round that fire at similar rates, with homogeneous rounds between them. It passes above
whichever way round the detectors run, and the detection there comes from the round profile, the
endpoint agreement and the state profile.

Two things follow from that symmetry, and both are in the rule. The comparison is on the paired
per shot difference against three of its own standard errors, not on the two rates against one
standard error of either: under the null of no difference a one sigma one sided test calls a fault
about one time in six whatever the sample size. And the check declines to run at all below 30
failures in an arm, where the comparison cannot discriminate.

Where it has power is a record whose noise changes through the run against a model that knows it.
Here is a run whose readout gets noisier round on round, with a model whose weights match, and the
same record handed over with its detector axis reversed.

In [6]:
from qb_compiler.record import RecordDem
from qb_compiler.record.controls import time_mirror

d, rounds, shots = 3, 6, 12000
rng = np.random.default_rng(1)
p_data = 0.03
p_meas = np.linspace(0.02, 0.30, rounds)  # readout gets noisier as the run goes on

state = np.zeros((shots, d), dtype=np.uint8)
stored = np.zeros((shots, rounds, d - 1), dtype=np.uint8)
for t in range(rounds):
    state ^= (rng.random((shots, d)) < p_data).astype(np.uint8)
    parity = state[:, :-1] ^ state[:, 1:]
    stored[:, t] = parity ^ (rng.random((shots, d - 1)) < p_meas[t]).astype(np.uint8)
state ^= (rng.random((shots, d)) < p_data).astype(np.uint8)
final_parity = (state[:, :-1] ^ state[:, 1:]).astype(np.uint8)
drifting = np.empty((shots, rounds + 1, d - 1), dtype=np.uint8)
drifting[:, 0] = stored[:, 0]
drifting[:, 1:rounds] = stored[:, 1:] ^ stored[:, :-1]
drifting[:, rounds] = final_parity ^ stored[:, rounds - 1]
drift_labels = state[:, 0].astype(np.uint8)

flat = build_repetition_dem(d, rounds, p_data=p_data, p_meas=float(p_meas[0]))
weights = np.asarray(flat.weights).copy()
position = d * (rounds + 1)  # the measurement mechanisms follow the data ones
for t in range(rounds):
    for _site in range(d - 1):
        weights[position] = -np.log(p_meas[t] / (1 - p_meas[t]))
        position += 1
matched = RecordDem(check_matrix=flat.check_matrix, observable=flat.observable, weights=weights)

as_recorded = validate(RecordSpec(detectors=drifting, labels=drift_labels, dem=matched))
reversed_axis = validate(
    RecordSpec(detectors=time_mirror(drifting), labels=drift_labels, dem=matched)
)
for name, report_ in (("as recorded", as_recorded), ("detector axis reversed", reversed_axis)):
    check = report_.check("time_mirror_control")
    print(f"{name:<24} {check.status}  critical {check.critical}")
    print(
        f"    {check.measured['logical_error_rate']:.5f} decoded, "
        f"{check.measured['mirrored_logical_error_rate']:.5f} reversed, "
        f"difference {check.measured['difference']:+.5f}, "
        f"limit {check.measured['limit']:.5f}, n {check.measured['n_used']}"
    )

as recorded              PASS  critical False
    0.06075 decoded, 0.12142 reversed, difference -0.06067, limit 0.00859, n 12000
detector axis reversed   FAIL  critical False
    0.12142 decoded, 0.06075 reversed, difference +0.06067, limit 0.00859, n 12000


## 5. The model fingerprint

Four numbers that do not depend on probabilities, ordering or which library built the model. Two
models with the same fingerprint are the same shape; two with different fingerprints are different
models, whatever the file names say. For the repetition code the counts are closed form, so the
fingerprint checks the builder against arithmetic rather than against itself.

In [7]:
from qb_compiler.record.dem import fingerprint
from qb_compiler.record.dem.repetition import expected_repetition_fingerprint

print(f"{'code':<10} {'mechanisms':>11} {'boundary':>9} {'closed form matches':>20}")
for d, rounds in ((3, 3), (5, 5), (7, 4), (11, 11)):
    measured = fingerprint(build_repetition_dem(d, rounds))
    closed = expected_repetition_fingerprint(d, rounds)
    label = f"d{d} r{rounds}"
    print(
        f"{label:<10} {measured['n_mechanisms']:>11} {measured['n_boundary']:>9} "
        f"{str(measured == closed):>20}"
    )

print()
print("d(r+1) data mechanisms, (d-1)r measurement mechanisms, 2(r+1) of them at a boundary")
for d, rounds in ((3, 3), (5, 5), (7, 4), (11, 11)):
    print(
        f"d{d} r{rounds}: {d * (rounds + 1)} + {(d - 1) * rounds} = "
        f"{d * (rounds + 1) + (d - 1) * rounds}, boundary {2 * (rounds + 1)}"
    )

code        mechanisms  boundary  closed form matches
d3 r3               18         8                 True
d5 r5               50        12                 True
d7 r4               59        10                 True
d11 r11            242        24                 True

d(r+1) data mechanisms, (d-1)r measurement mechanisms, 2(r+1) of them at a boundary
d3 r3: 12 + 6 = 18, boundary 8
d5 r5: 30 + 20 = 50, boundary 12
d7 r4: 35 + 24 = 59, boundary 10
d11 r11: 132 + 110 = 242, boundary 24


## 6. Writing the record down, and reading it back

One file, named keys, declared dtypes. The reader refuses a wrong dtype or a wrong shape rather
than coercing it, because a record quietly widened from uint8 still decodes and the number it
produces is wrong where nothing later can see it.

In [8]:
import subprocess
import tempfile
from pathlib import Path

from qb_compiler.record.loaders import read_npz, write_npz

workspace = Path(tempfile.mkdtemp(prefix="qbc-record-demo-"))
write_npz(workspace / "good.npz", spec)
write_npz(workspace / "mirrored.npz", mirrored)

restored = read_npz(workspace / "good.npz")
again = validate(restored)
print("same verdict after a round trip:", again.passed == report.passed)
print(
    "same status on every check:",
    [c.status for c in again.checks] == [c.status for c in report.checks],
)
print("detectors identical:", np.array_equal(restored.detectors, spec.detectors))

same verdict after a round trip: True
same status on every check: True
detectors identical: True


In [9]:
broken = workspace / "widened.npz"
with broken.open("wb") as handle:
    np.savez_compressed(
        handle,
        dets=np.asarray(spec.detectors, dtype=np.int64),
        labels=np.asarray(spec.labels, dtype=np.uint8),
    )
try:
    read_npz(broken)
except ValueError as exc:
    print("refused:", str(exc).split(": ", 1)[1][:150])

refused: 'dets' has dtype int64, expected uint8. This reader does not coerce: a record quietly widened or narrowed still decodes and produces a wrong number no


The same checks from the command line, summary to stderr and JSON to stdout.

In [10]:
import json

for name in ("good.npz", "mirrored.npz"):
    done = subprocess.run(
        ["qbc", "record", "validate", str(workspace / name)],
        capture_output=True,
        text=True,
    )
    verdict = json.loads(done.stdout)
    failed = [c["name"] for c in verdict["checks"] if c["status"] == "FAIL"]
    print(f"{name:<14} exit {done.returncode}  passed {verdict['passed']}  failed {failed}")
print()
print("exit 0 every critical check passed, 2 a critical check failed, 3 could not run, 1 error")

good.npz       exit 0  passed True  failed []


mirrored.npz   exit 2  passed False  failed ['round_profile', 'endpoint_agreement', 'state_profile']

exit 0 every critical check passed, 2 a critical check failed, 3 could not run, 1 error


## 7. Optional: a record from hardware

This section runs only when `QB_RECORD_DATA` points at a directory holding `ibm_fez`. It loads one
regime both ways round and prints the per round detector rates and the check table for each.

In [11]:
import os

root = os.environ.get("QB_RECORD_DATA")
if not root:
    print("record data not present, section skipped")
else:
    from qb_compiler.record.loaders import ibm_fez_repetition

    for label, stored_order in (("corrected", False), ("as stored", True)):
        loaded = ibm_fez_repetition.load(
            Path(root) / "ibm_fez", d=5, r=5, basis="Z", reverse_rounds=not stored_order
        )
        real = RecordSpec(
            detectors=loaded.detectors,
            labels=loaded.labels,
            groups=loaded.groups,
            raw_syndromes=loaded.raw_syndromes,
            final_parity=loaded.final_parity,
            logical_state=loaded.logical_state,
            dem=build_repetition_dem(5, 5, p_data=0.01, p_meas=0.01),
            meta=loaded.meta,
        )
        real_report = validate(real)
        rates = real_report.check("round_profile").measured["rate_per_round"]
        error_rate = (decode_records(real.dem, real.detector_matrix()) != real.labels).mean()
        print(f"IBM Fez d5 r5, {label}")
        print("  detector rate per round:", [round(float(x), 4) for x in rates])
        print(f"  logical error rate: {error_rate:.4f}")
        print(f"  passed: {real_report.passed}")
        for item in real_report.checks:
            if item.status != "SKIP":
                print(f"    {item.name:<22} {item.status}")
        print()

IBM Fez d5 r5, corrected
  detector rate per round: [0.0516, 0.0989, 0.0998, 0.099, 0.1006, 0.0769]
  logical error rate: 0.0162
  passed: True
    round_profile          PASS
    endpoint_agreement     PASS
    event_density          PASS
    time_mirror_control    PASS
    state_profile          PASS
    dem_fingerprint        PASS



IBM Fez d5 r5, as stored
  detector rate per round: [0.1919, 0.1006, 0.099, 0.0998, 0.0989, 0.2017]
  logical error rate: 0.0383
  passed: False
    round_profile          FAIL
    endpoint_agreement     FAIL
    event_density          PASS
    time_mirror_control    PASS
    state_profile          FAIL
    dem_fingerprint        PASS



In [12]:
import shutil

shutil.rmtree(workspace)
print("temporary files removed")

temporary files removed


## What this tells you, and what it does not

A record that passes these checks is correctly built. That is the whole claim. It is not complete,
and nothing here says it is: a correctly built record can still carry structure that the decoder
reading it does not use, which is a different question and a different measurement.

A passing report is also not a statement about the decoder. None of these checks asks whether a
decoder is any good. They ask whether the thing it was handed is the thing the experiment
produced.